# MELD 7-Class Emotion — UFEN+MTFN with MultiEMO Features (Exp 9.1)

**Goal:** Apply the proposal's UFEN+MTFN architecture to stronger pre-extracted features from the MultiEMO (ACL 2023) release, via a Multi-Head Feature Tokenization (MHFT) front-end that bridges utterance-level vectors into UFEN's sequence-aware pipeline.

**Exp 9.1 changes over Exp 9 (to close the dev→test gap):**
1. **Speaker-disjoint dev split** — instead of random 90/10 split of `trainVid`, we hold out whole dialogues dominated by a small set of rarer speakers. This reduces speaker leakage between train and dev so the dev F1 correlates better with test F1.
2. **Capped class weights** — inverse-frequency weights for rare classes (Fear, Disgust) are capped at `1.5`. In Exp 9 the uncapped Fear weight of 2.24 made the model over-fire on Fear (test recall 52% but precision 0.12); the cap prevents this.
3. **Top-K checkpoint ensemble** — instead of selecting the single best epoch on a noisy dev set, we save the top-5 epochs by dev F1w and average their test-time softmax probabilities. This reduces epoch-selection variance.

**What stays identical to the proposal (unchanged from Exp 9):**
- ✅ **UFEN** — BiGRU + parallel Conv1D branches + self-attention + unpool + sum + LayerNorm + mean-pool + pred_head
- ✅ **MTFN** — 6 directed cross-modal attention pairs + encoder + decoder + dual prediction heads
- ✅ **Multi-task loss** — 5 simultaneous CE losses
- ✅ **MHFT front-end** — K=8 learned tokens per modality

**Setup on Kaggle:**
1. New Notebook → Settings → Accelerator → **GPU T4 x1**
2. Upload this notebook → Run All
3. No dataset attachment needed — features are downloaded at runtime (~213 MB)

In [ ]:
!pip install -q tqdm scikit-learn

In [ ]:
import os, sys, math, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.optim import Adam
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from tqdm import tqdm
from types import SimpleNamespace

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB')

## 1. Download MultiEMO Features (~213 MB, one-time)

In [ ]:
FEAT_DIR = '/kaggle/working/meld_feats'
os.makedirs(FEAT_DIR, exist_ok=True)

BASE = 'https://github.com/LuckyDaydreamer/MultiEMO/raw/main/Data/MELD'
FILES = ['AudioFeatures.pkl', 'VisualFeatures.pkl', 'TextFeatures.pkl', 'Speakers.pkl']

for fn in FILES:
    out = f'{FEAT_DIR}/{fn}'
    if os.path.exists(out):
        print(f'  {fn}: already downloaded ({os.path.getsize(out)/1e6:.1f} MB)')
        continue
    print(f'Downloading {fn}...')
    !wget -q -O {out} {BASE}/{fn}
    print(f'  -> {os.path.getsize(out)/1e6:.1f} MB')

!ls -lh {FEAT_DIR}

## 2. Load & Inspect Features

In [ ]:
def load_pkl(name):
    with open(f'{FEAT_DIR}/{name}', 'rb') as f:
        return pickle.load(f)

textFeatures   = load_pkl('TextFeatures.pkl')
audioFeatures  = load_pkl('AudioFeatures.pkl')
visualFeatures = load_pkl('VisualFeatures.pkl')
speakersData   = load_pkl('Speakers.pkl')

def describe(name, obj, depth=0, max_depth=3):
    indent = '  ' * depth
    if isinstance(obj, dict):
        print(f'{indent}{name}: dict, {len(obj)} keys')
        if depth < max_depth:
            sample_key = next(iter(obj.keys()))
            print(f'{indent}  sample key: {repr(sample_key)[:60]}')
            describe('[val]', obj[sample_key], depth + 1, max_depth)
    elif isinstance(obj, (list, tuple)):
        print(f'{indent}{name}: {type(obj).__name__}, len={len(obj)}')
        if len(obj) > 0 and depth < max_depth:
            describe('[0]', obj[0], depth + 1, max_depth)
    elif hasattr(obj, 'shape'):
        print(f'{indent}{name}: {type(obj).__name__} shape={obj.shape} dtype={getattr(obj, "dtype", "?")}')
    else:
        print(f'{indent}{name}: {type(obj).__name__} = {repr(obj)[:80]}')

print('=== TextFeatures ==='); describe('TextFeatures', textFeatures)
print('\n=== AudioFeatures ==='); describe('AudioFeatures', audioFeatures)
print('\n=== VisualFeatures ==='); describe('VisualFeatures', visualFeatures)
print('\n=== Speakers.pkl ==='); describe('Speakers', speakersData)

## 3. Parse Splits & Flatten to Per-Utterance Samples

In [ ]:
def parse_speakers(obj):
    if isinstance(obj, dict):
        return (obj.get('videoSpeakers') or obj.get('speakers'),
                obj.get('videoLabels') or obj.get('labels'),
                obj.get('trainVid')    or obj.get('train'),
                obj.get('validVid')    or obj.get('valid') or obj.get('devVid') or obj.get('dev'),
                obj.get('testVid')     or obj.get('test'))
    if isinstance(obj, (list, tuple)):
        # MELD_features_raw standard format (MMGCN / MM-DFN / DialogueGCN):
        # (videoIDs, videoSpeakers, videoLabels, videoText, videoAudio,
        #  videoVisual, videoSentence, trainVid, testVid, validVid)
        if len(obj) == 10:
            return obj[1], obj[2], obj[7], obj[9], obj[8]
        if len(obj) == 5:
            return obj[0], obj[1], obj[2], obj[4], obj[3]
        if len(obj) == 4:
            return obj[0], obj[1], obj[2], None, obj[3]
    raise ValueError(f'Unrecognised Speakers.pkl format: {type(obj)} len={len(obj) if hasattr(obj, "__len__") else "?"}')

videoSpeakers, videoLabels, trainVid, validVid, testVid = parse_speakers(speakersData)

def _sz(x): return len(x) if x is not None and hasattr(x, '__len__') else 'None'
print(f'videoSpeakers: {_sz(videoSpeakers)} dialogues')
print(f'videoLabels  : {_sz(videoLabels)} dialogues')
print(f'trainVid     : {_sz(trainVid)}')
print(f'validVid     : {_sz(validVid)}')
print(f'testVid      : {_sz(testVid)}')

# Peek at speaker format (could be strings, ints, or one-hot vectors)
sample_dia = next(iter(trainVid))
sample_speakers = videoSpeakers[sample_dia]
print(f'\nSample dialogue {sample_dia!r}: {len(sample_speakers)} utterances')
print(f'  First speaker entry: {repr(sample_speakers[0])[:100]}')
print(f'  Type: {type(sample_speakers[0]).__name__}')

In [ ]:
EMOTION_LABELS = ['Neutral', 'Surprise', 'Fear', 'Sadness', 'Joy', 'Disgust', 'Anger']
NUM_CLASSES = 7

def to_numpy(x):
    if isinstance(x, np.ndarray):  return x.astype(np.float32)
    if hasattr(x, 'numpy'):        return x.detach().cpu().numpy().astype(np.float32)
    return np.asarray(x, dtype=np.float32)

# ========================================================================
# NEW: Speaker-disjoint dev split
# ------------------------------------------------------------------------
# Friends has ~6 main characters who dominate most dialogues. A random
# dialogue-level split still puts all characters in both sets, so the
# model memorises speaker prosody and the dev signal becomes unreliable.
# Instead we compute the DOMINANT speaker per dialogue and hold out whole
# speaker groups as dev — starting with the rarest speakers until we hit
# ~10% of training dialogues.
# ========================================================================
from collections import Counter, defaultdict

def get_dominant_speaker(speakers_seq):
    """Return the most common speaker id in a per-utterance speaker list.
    Handles strings, ints, and one-hot/probability vectors."""
    if not speakers_seq:
        return None
    normalized = []
    for s in speakers_seq:
        if isinstance(s, (list, tuple, np.ndarray)) and not isinstance(s, str):
            arr = np.asarray(s)
            if arr.ndim >= 1:
                s = int(np.argmax(arr.flatten()))
        normalized.append(s)
    return Counter(normalized).most_common(1)[0][0]

def speaker_disjoint_split(trainVid, target_frac=0.10):
    """Hold out whole dialogues keyed by rarest dominant speakers."""
    dia_dom = {d: get_dominant_speaker(videoSpeakers[d]) for d in trainVid}
    by_speaker = defaultdict(list)
    for d, spk in dia_dom.items():
        by_speaker[spk].append(d)

    # Sort by popularity ascending — we fill dev from rarest speakers first
    sorted_spk = sorted(by_speaker.items(), key=lambda kv: len(kv[1]))
    target = int(round(target_frac * len(trainVid)))
    dev_set, train_set = set(), set(trainVid)
    held_speakers = []
    for spk, dias in sorted_spk:
        if len(dev_set) >= target:
            break
        dev_set.update(dias)
        held_speakers.append((spk, len(dias)))
        for d in dias:
            train_set.discard(d)
    return train_set, dev_set, held_speakers

if validVid is None or len(validVid) == 0:
    print('validVid missing — creating speaker-disjoint dev split from trainVid')
    trainVid, validVid, held = speaker_disjoint_split(trainVid, target_frac=0.10)
    print(f'  held-out speakers ({len(held)}): first 10 = {held[:10]}')
    print(f'  new trainVid: {len(trainVid)}, new validVid: {len(validVid)}')
else:
    print('Using native validVid — no split needed')

def flatten_split(dialogue_ids):
    out, missing = [], 0
    for dia_id in dialogue_ids:
        if dia_id not in videoLabels or dia_id not in textFeatures:
            missing += 1
            continue
        labels_d = videoLabels[dia_id]
        text_d   = textFeatures[dia_id]
        audio_d  = audioFeatures[dia_id]
        video_d  = visualFeatures[dia_id]
        for i in range(len(labels_d)):
            out.append({
                'text':  to_numpy(text_d[i]),
                'audio': to_numpy(audio_d[i]),
                'video': to_numpy(video_d[i]),
                'label': int(labels_d[i]),
            })
    if missing: print(f'  missing dialogues skipped: {missing}')
    return out

train_data = flatten_split(trainVid)
dev_data   = flatten_split(validVid)
test_data  = flatten_split(testVid)
print(f'\nTrain={len(train_data)}, Dev={len(dev_data)}, Test={len(test_data)}')

def count_labels(data):
    c = np.zeros(NUM_CLASSES, dtype=int)
    for s in data: c[s['label']] += 1
    return c

print('\nLabel distribution:')
print(f'  {"":<10} ' + ' '.join(f'{e[:5]:>7}' for e in EMOTION_LABELS))
for name, d in [('train', train_data), ('dev', dev_data), ('test', test_data)]:
    c = count_labels(d)
    print(f'  {name:<10} ' + ' '.join(f'{x:>7d}' for x in c))

TEXT_DIM  = train_data[0]['text'].shape[-1]
AUDIO_DIM = train_data[0]['audio'].shape[-1]
VIDEO_DIM = train_data[0]['video'].shape[-1]
print(f'\nFeature dims — text:{TEXT_DIM}, audio:{AUDIO_DIM}, video:{VIDEO_DIM}')

## 4. Config

In [ ]:
config = SimpleNamespace(
    modalities=['text', 'audio', 'video'],

    # Feature dims (inferred from MultiEMO pkls)
    text_dim  = TEXT_DIM,
    audio_dim = AUDIO_DIM,
    video_dim = VIDEO_DIM,

    # MHFT front-end (NEW)
    n_tokens = 8,           # number of synthetic tokens per utterance

    # UFEN (UNCHANGED from Exp 1-6)
    d_m          = 128,
    conv_dim     = 64,
    n_layers     = 2,
    kernel_sizes = [1, 5],
    self_att_heads = 1,

    # MTFN (UNCHANGED from Exp 1-6)
    d_ff             = 256,
    cross_att_heads  = 4,
    att_dropout      = 0.2,
    dropout          = 0.1,
    num_classes      = NUM_CLASSES,

    # Training
    batch_size       = 64,
    lr               = 1e-3,
    epochs           = 60,
    early_stop       = 12,
    grad_clip        = 1.0,
    label_smoothing  = 0.1,
    use_class_weights= True,
    use_lr_scheduler = True,
)
print(config)

## 5. Model — MHFT + UFEN + MTFN

**MHFT (NEW):** Projects `(B, dim)` → `(B, K=8, d_m)` via K independent learned heads. Acts as a feature tokenizer that creates a synthetic sequence for UFEN to process.

**UFEN (UNCHANGED):** BiGRU → N parallel [Conv1D → Self-Att → Unpool] → sum → mean-pool → pred_head. Now operates on the K MHFT tokens instead of word-aligned frames.

**MTFN (UNCHANGED):** 6 directed cross-modal attention pairs → stack → encoder → decoder → fusion+recon heads.

**Multi-task loss (UNCHANGED):** Sum of 5 CE losses (text, audio, video, fusion, recon).

In [ ]:
def masked_mean(x, mask=None):
    if mask is None:
        return x.mean(dim=1)
    valid = (~mask).float().unsqueeze(-1)
    return (x * valid).sum(dim=1) / valid.sum(dim=1).clamp(min=1.0)


# ============================================================================
# NEW: Multi-Head Feature Tokenization (MHFT)
# ----------------------------------------------------------------------------
# Projects an utterance-level vector into K learned token embeddings, creating
# a synthetic length-K sequence that UFEN can process unchanged. Each head
# learns to capture a different aspect of the input feature (analogous to
# multi-head attention).
# ============================================================================
class FeatureTokenizer(nn.Module):
    def __init__(self, input_dim, d_m, n_tokens=8, dropout=0.1):
        super().__init__()
        self.n_tokens = n_tokens
        self.d_m      = d_m
        self.heads    = nn.Linear(input_dim, n_tokens * d_m)
        self.norm     = nn.LayerNorm(d_m)
        self.dropout  = nn.Dropout(dropout)
        # Learned positional embedding for the K tokens
        self.pos_emb  = nn.Parameter(torch.zeros(1, n_tokens, d_m))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

    def forward(self, x):                       # x: (B, input_dim)
        h = self.heads(x)                       # (B, K * d_m)
        h = h.view(x.size(0), self.n_tokens, self.d_m)
        h = self.norm(h) + self.pos_emb
        return self.dropout(h)                  # (B, K, d_m)


# ============================================================================
# UFEN — UNCHANGED from Exp 1-6 (phase2/model.py)
# ============================================================================
class UFEN(nn.Module):
    def __init__(self, input_dim, d_m, num_classes, n_layers=2, kernel_sizes=None,
                 conv_dim=64, n_att_heads=1, dropout=0.1):
        super().__init__()
        kernel_sizes = kernel_sizes or [1, 3]
        assert len(kernel_sizes) == n_layers
        self.bigru = nn.GRU(input_dim, d_m // 2, batch_first=True, bidirectional=True)
        self.convs = nn.ModuleList([
            nn.Conv1d(d_m, conv_dim, k, padding=k // 2) for k in kernel_sizes
        ])
        self.self_atts = nn.ModuleList([
            nn.MultiheadAttention(conv_dim, n_att_heads, dropout=dropout, batch_first=True)
            for _ in range(n_layers)
        ])
        self.unpools = nn.ModuleList([nn.Linear(conv_dim, d_m) for _ in range(n_layers)])
        self.layer_norm = nn.LayerNorm(d_m)
        self.dropout    = nn.Dropout(dropout)
        self.pred_head  = nn.Linear(d_m, num_classes)

    def forward(self, x, mask=None):            # x: (B, T, input_dim)
        x, _ = self.bigru(x)                    # (B, T, d_m)
        if mask is not None:
            x = x.masked_fill(mask.unsqueeze(-1), 0.0)
        branches = []
        for conv, sa, up in zip(self.convs, self.self_atts, self.unpools):
            c = F.relu(conv(x.transpose(1, 2)).transpose(1, 2))
            if mask is not None:
                c = c.masked_fill(mask.unsqueeze(-1), 0.0)
            att, _ = sa(c, c, c, key_padding_mask=mask)
            branches.append(up(c * att))
        fusion = self.dropout(self.layer_norm(sum(branches)))
        if mask is not None:
            fusion = fusion.masked_fill(mask.unsqueeze(-1), 0.0)
        return fusion, self.pred_head(masked_mean(fusion, mask))


# ============================================================================
# MTFN building blocks — UNCHANGED from Exp 1-6
# ============================================================================
class CrossModalAttention(nn.Module):
    def __init__(self, d_m, n_heads, att_dropout=0.2):
        super().__init__()
        self.mha = nn.MultiheadAttention(d_m, n_heads, dropout=att_dropout, batch_first=True)
        self.layer_norm = nn.LayerNorm(d_m)
    def forward(self, x_i, x_j, mask_i=None, mask_j=None):
        out, _ = self.mha(x_i, x_j, x_j, key_padding_mask=mask_j)
        return self.layer_norm(out)


class EncoderBlock(nn.Module):
    def __init__(self, d_m, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_m, n_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d_m, d_ff), nn.ReLU(), nn.Linear(d_ff, d_m))
        self.norm1 = nn.LayerNorm(d_m); self.norm2 = nn.LayerNorm(d_m)
        self.drop  = nn.Dropout(dropout)
    def forward(self, x):
        a, _ = self.self_attn(x, x, x)
        x = self.norm1(x + self.drop(a))
        return self.norm2(x + self.drop(self.ff(x)))


class DecoderBlock(nn.Module):
    def __init__(self, d_m, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(d_m, n_heads, dropout=dropout, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d_m, d_ff), nn.ReLU(), nn.Linear(d_ff, d_m))
        self.norm1 = nn.LayerNorm(d_m); self.norm2 = nn.LayerNorm(d_m)
        self.drop  = nn.Dropout(dropout)
    def forward(self, x, memory):
        a, _ = self.cross_attn(x, memory, memory)
        x = self.norm1(x + self.drop(a))
        return self.norm2(x + self.drop(self.ff(x)))


class MTFN(nn.Module):
    def __init__(self, d_m, num_classes, modalities, n_cross_heads=4, d_ff=256,
                 dropout=0.1, att_dropout=0.2):
        super().__init__()
        self.modalities = modalities
        self.projs = nn.ModuleDict({m: nn.Linear(d_m, d_m) for m in modalities})
        self.cross_atts = nn.ModuleDict({
            f'{mi}2{mj}': CrossModalAttention(d_m, n_cross_heads, att_dropout)
            for mi in modalities for mj in modalities if mi != mj
        })
        n_pairs = len(modalities) * (len(modalities) - 1)
        self.encoder     = EncoderBlock(d_m, n_cross_heads, d_ff, dropout)
        self.decoder     = DecoderBlock(d_m, n_cross_heads, d_ff, dropout)
        self.pred_fusion = nn.Linear(n_pairs * d_m, num_classes)
        self.pred_recon  = nn.Linear(d_m, num_classes)

    def forward(self, feats, masks):
        proj   = {m: self.projs[m](feats[m]) for m in self.modalities}
        tokens = []
        for mi in self.modalities:
            for mj in self.modalities:
                if mi != mj:
                    ca = self.cross_atts[f'{mi}2{mj}'](proj[mi], proj[mj], masks.get(mi), masks.get(mj))
                    tokens.append(masked_mean(ca, masks.get(mi)))
        tokens    = torch.stack(tokens, dim=1)
        y_m       = self.pred_fusion(tokens.reshape(tokens.size(0), -1))
        enc_out   = self.encoder(tokens)
        dec_out   = self.decoder(tokens, enc_out)
        y_m_prime = self.pred_recon(dec_out.mean(dim=1))
        return y_m, y_m_prime


# ============================================================================
# Top-level model: MHFT (NEW) → UFEN (UNCHANGED) → MTFN (UNCHANGED)
# ============================================================================
class MultiTaskModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.modalities = cfg.modalities
        C = cfg.num_classes
        dims = {'text': cfg.text_dim, 'audio': cfg.audio_dim, 'video': cfg.video_dim}

        # MHFT (NEW) — one tokenizer per modality
        self.tokenizers = nn.ModuleDict({
            m: FeatureTokenizer(dims[m], cfg.d_m, cfg.n_tokens, cfg.dropout)
            for m in self.modalities
        })

        # UFEN (UNCHANGED) — one per modality
        # Note: input_dim = d_m because MHFT outputs d_m-dim tokens
        self.ufens = nn.ModuleDict({
            m: UFEN(input_dim=cfg.d_m, d_m=cfg.d_m, num_classes=C,
                    n_layers=cfg.n_layers, kernel_sizes=cfg.kernel_sizes,
                    conv_dim=cfg.conv_dim, n_att_heads=cfg.self_att_heads,
                    dropout=cfg.dropout)
            for m in self.modalities
        })

        # MTFN (UNCHANGED)
        self.mtfn = MTFN(cfg.d_m, C, self.modalities, cfg.cross_att_heads, cfg.d_ff,
                         cfg.dropout, cfg.att_dropout) if len(self.modalities) >= 2 else None

    def forward(self, text, audio, video):
        inputs = {'text': text, 'audio': audio, 'video': video}
        feats, masks, preds = {}, {}, {}

        for m in self.modalities:
            tokens = self.tokenizers[m](inputs[m])              # (B, K, d_m)
            feat, y = self.ufens[m](tokens, mask=None)          # (B, K, d_m), (B, C)
            feats[m] = feat
            masks[m] = None                                     # no padding (synthetic seq)
            preds[m] = y

        if self.mtfn is not None:
            y_m, y_m_prime = self.mtfn(feats, masks)
            preds['fusion'], preds['recon'] = y_m, y_m_prime
        else:
            sole = preds[self.modalities[0]]
            preds['fusion'] = preds['recon'] = sole

        return preds

## 6. Dataset, Utilities & Training Loop

In [ ]:
class MELDFeatureDataset(Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, i):
        s = self.data[i]
        return (
            torch.from_numpy(s['text']),
            torch.from_numpy(s['audio']),
            torch.from_numpy(s['video']),
            torch.tensor(s['label'], dtype=torch.long),
        )

def make_loader(data, shuffle):
    return DataLoader(MELDFeatureDataset(data), batch_size=config.batch_size,
                      shuffle=shuffle, num_workers=2, pin_memory=True)

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ========================================================================
# NEW: Capped class weights
# ------------------------------------------------------------------------
# Inverse-frequency weights gave rare classes (Fear, Disgust) weights of
# ~2.2 and ~2.4, which caused the model to over-fire on them on test
# (Fear: 52% recall, 12% precision). Capping at 1.5 keeps rare classes
# up-weighted but prevents pathological aggression.
# ========================================================================
def compute_class_weights(data, cap=1.5):
    counts = np.zeros(NUM_CLASSES)
    for s in data: counts[s['label']] += 1
    w = 1.0 / np.clip(counts, 1, None)
    w = w / w.mean()
    if cap is not None:
        w = np.minimum(w, cap)
    return torch.FloatTensor(w)

def compute_metrics(p, l):
    return {
        'Accuracy':    accuracy_score(l, p) * 100,
        'F1_weighted': f1_score(l, p, average='weighted', zero_division=0) * 100,
        'F1_macro':    f1_score(l, p, average='macro',    zero_division=0) * 100,
    }

def evaluate(model, loader):
    model.eval()
    use_amp = torch.cuda.is_available()
    all_p, all_l = [], []
    with torch.no_grad():
        for text, audio, video, labels in loader:
            text, audio, video = text.to(device), audio.to(device), video.to(device)
            with autocast('cuda', enabled=use_amp):
                preds = model(text, audio, video)
            all_p.append(preds['recon'].argmax(1).cpu().numpy())
            all_l.append(labels.numpy())
    p, l = np.concatenate(all_p), np.concatenate(all_l)
    return compute_metrics(p, l), p, l

# ========================================================================
# NEW: Top-K checkpoint ensemble evaluation
# ------------------------------------------------------------------------
# Instead of picking the single best-dev checkpoint (which is noisy), we
# keep the top-K by dev F1w, run each on test, and average their softmax
# probabilities. This smooths out epoch-selection variance.
# ========================================================================
@torch.no_grad()
def ensemble_evaluate(model, loader, ckpt_paths):
    use_amp = torch.cuda.is_available()
    agg_probs = None
    all_labels = None
    for path in ckpt_paths:
        model.load_state_dict(torch.load(path, map_location=device))
        model.eval()
        probs_list, labels_list = [], []
        for text, audio, video, labels in loader:
            text, audio, video = text.to(device), audio.to(device), video.to(device)
            with autocast('cuda', enabled=use_amp):
                preds = model(text, audio, video)
            probs_list.append(F.softmax(preds['recon'].float(), dim=-1).cpu())
            labels_list.append(labels)
        probs_ckpt = torch.cat(probs_list, dim=0)
        if agg_probs is None:
            agg_probs  = probs_ckpt
            all_labels = torch.cat(labels_list)
        else:
            agg_probs = agg_probs + probs_ckpt
    agg_probs /= len(ckpt_paths)
    preds  = agg_probs.argmax(dim=-1).numpy()
    labels = all_labels.numpy()
    return compute_metrics(preds, labels), preds, labels

def print_full_report(p, l):
    print('\n--- Classification Report ---')
    print(classification_report(l, p, target_names=EMOTION_LABELS, zero_division=0))
    print('--- Confusion Matrix ---')
    cm = confusion_matrix(l, p)
    print('          ' + ' '.join(f'{EMOTION_LABELS[i][:4]:>5}' for i in range(NUM_CLASSES)))
    for i in range(NUM_CLASSES):
        print(f'{EMOTION_LABELS[i]:<10}' + ' '.join(f'{cm[i,j]:5d}' for j in range(NUM_CLASSES)))

In [ ]:
set_seed(42)

train_loader = make_loader(train_data, shuffle=True)
dev_loader   = make_loader(dev_data,   shuffle=False)
test_loader  = make_loader(test_data,  shuffle=False)

class_weights = compute_class_weights(train_data).to(device) if config.use_class_weights else None
if class_weights is not None:
    print(f'Class weights: {class_weights.cpu().numpy().round(3)}')

model = MultiTaskModel(config).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params:,}')

# Per-component param counts
n_tok = sum(p.numel() for p in model.tokenizers.parameters() if p.requires_grad)
n_uf  = sum(p.numel() for p in model.ufens.parameters() if p.requires_grad)
n_mt  = sum(p.numel() for p in model.mtfn.parameters() if p.requires_grad) if model.mtfn else 0
print(f'  MHFT (new) : {n_tok:>8,}')
print(f'  UFEN       : {n_uf:>8,}')
print(f'  MTFN       : {n_mt:>8,}')

optimizer = Adam(model.parameters(), lr=config.lr)
scheduler = LambdaLR(
    optimizer,
    lambda t: max(0., 0.5 * (1 + math.cos(math.pi * t / max(1, config.epochs))))
) if config.use_lr_scheduler else None

ce_loss = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=config.label_smoothing)
print('Loss: CrossEntropyLoss')

In [ ]:
# ========================================================================
# Training loop with top-K checkpoint tracking
# ========================================================================
import heapq

CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
TOP_K = 5

# Min-heap of (dev_f1w, epoch, ckpt_path) — root is the WORST of the top-K
topk_heap = []

best_dev_f1, best_epoch = 0.0, 0
use_amp = torch.cuda.is_available()
scaler  = GradScaler('cuda', enabled=use_amp)

def save_ckpt_topk(epoch, dev_f1w):
    """Maintain top-K checkpoints by dev F1w."""
    ckpt_path = f'{CKPT_DIR}/ckpt_ep{epoch:02d}.pt'
    if len(topk_heap) < TOP_K:
        torch.save(model.state_dict(), ckpt_path)
        heapq.heappush(topk_heap, (dev_f1w, epoch, ckpt_path))
        return True
    elif dev_f1w > topk_heap[0][0]:
        # Evict the worst
        _, _, worst_path = heapq.heappop(topk_heap)
        try: os.remove(worst_path)
        except FileNotFoundError: pass
        torch.save(model.state_dict(), ckpt_path)
        heapq.heappush(topk_heap, (dev_f1w, epoch, ckpt_path))
        return True
    return False

for epoch in range(1, config.epochs + 1):
    model.train()
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f'Epoch {epoch:02d}', leave=False, mininterval=1.0)
    for text, audio, video, labels in pbar:
        text, audio, video, labels = text.to(device), audio.to(device), video.to(device), labels.to(device)
        with autocast('cuda', enabled=use_amp):
            preds = model(text, audio, video)
            loss = sum(ce_loss(preds[k], labels) for k in list(config.modalities) + ['fusion', 'recon'])
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    if scheduler is not None:
        scheduler.step()

    avg_loss = total_loss / len(train_loader)
    dev_metrics, _, _ = evaluate(model, dev_loader)
    print(f"Epoch {epoch:02d} | loss={avg_loss:.4f} | Acc={dev_metrics['Accuracy']:.1f} | "
          f"F1w={dev_metrics['F1_weighted']:.1f} | F1m={dev_metrics['F1_macro']:.1f}")

    # Update top-K and track absolute best for early stopping
    saved = save_ckpt_topk(epoch, dev_metrics['F1_weighted'])
    if dev_metrics['F1_weighted'] > best_dev_f1:
        best_dev_f1, best_epoch = dev_metrics['F1_weighted'], epoch
        print(f'  -> New best Dev F1w={best_dev_f1:.2f} — saved.')
    elif saved:
        print(f'  -> Added to top-{TOP_K} (rank updated, Dev F1w={dev_metrics["F1_weighted"]:.2f})')
    elif (epoch - best_epoch) >= config.early_stop:
        print(f'Early stopping at epoch {epoch} (best: {best_epoch})')
        break

# ========================================================================
# Final evaluation: best-single AND top-K ensemble
# ========================================================================
# Sort top-K by dev F1w descending for reporting
topk_sorted = sorted(topk_heap, key=lambda x: -x[0])
print(f'\nTop-{TOP_K} checkpoints by Dev F1w:')
for rank, (f1w, ep, path) in enumerate(topk_sorted, 1):
    print(f'  #{rank}: epoch {ep:02d}  dev F1w={f1w:.2f}')

# Best single checkpoint
print(f'\n--- Single best checkpoint (epoch {best_epoch}) ---')
best_path = next(p for f, e, p in topk_sorted if e == best_epoch)
model.load_state_dict(torch.load(best_path, map_location=device))
single_metrics, single_preds, single_labels = evaluate(model, test_loader)
print(f"Accuracy       : {single_metrics['Accuracy']:.1f}")
print(f"F1 (weighted)  : {single_metrics['F1_weighted']:.1f}")
print(f"F1 (macro)     : {single_metrics['F1_macro']:.1f}")

# Top-K ensemble
print(f'\n--- Top-{TOP_K} ensemble (softmax avg of epochs {[e for _,e,_ in topk_sorted]}) ---')
ensemble_metrics, ens_preds, ens_labels = ensemble_evaluate(
    model, test_loader, [p for _, _, p in topk_sorted]
)

print('\n========== Test Results (Top-K Ensemble) ==========')
print(f"Accuracy       : {ensemble_metrics['Accuracy']:.1f}")
print(f"F1 (weighted)  : {ensemble_metrics['F1_weighted']:.1f}")
print(f"F1 (macro)     : {ensemble_metrics['F1_macro']:.1f}")
print('===================================================')
print_full_report(ens_preds, ens_labels)